# 🎓 Scholarship Recommendation System — Hybrid (Semantic + WSM/MCDM)
**Kelompok G — Machine Learning SD-A2**

### Arsitektur Baru
```
CV PDF  →  Text Extraction  →  Feature Extraction
                                  (IPK, Org, Prestasi)
                                        ↓
                          ┌─────────────────────────┐
                          │  Semantic Matching       │
                          │  (BM25 + Sentence-BERT)  │
                          └────────────┬────────────┘
                                       │
                          ┌────────────┴────────────┐
                          │  + WSM / MCDM Score      │
                          └────────────┬────────────┘
                                       ↓
                          Final Recommendation Score
                                       ↓
                               Ranking Beasiswa
```
**Pipeline:**
1. Ekstrak teks CV dari PDF
2. Feature Extraction: IPK, Organisasi, Prestasi
3. Analisis syarat beasiswa → Bobot WSM otomatis
4. BM25 + Sentence Transformer → Semantic Score
5. Hybrid Fusion: `FinalScore = α × Similarity + (1−α) × WSM`
6. Ranking beasiswa per user

In [ ]:
# Install semua library yang diperlukan
!pip install requests beautifulsoup4 pandas lxml -q
!pip install pypdf sentence-transformers rank_bm25 scikit-learn numpy -q

import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
import re
import json
import os
from datetime import datetime
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity

print('✅ Semua library berhasil diimport')

In [ ]:
# ============================================================
# DATA 5 BEASISWA AKTIF - PERIODE JUNI 2026
# Sumber: indbeasiswa.com, situs resmi masing-masing beasiswa
# ============================================================

data_beasiswa = [
    {
        'id': 'B001',
        'nama_beasiswa': 'Djarum Beasiswa Plus 2026',
        'penyelenggara': 'Djarum Foundation',
        'jenjang': 'S1,D4',
        'ipk_min': 3.00,
        'bidang': 'Semua Jurusan',
        'syarat': 'Semester 4, IPK min 3.00 di sem 3, aktif organisasi, tidak menerima beasiswa lain, kuliah di PT mitra Djarum',
        'benefit': 'Rp1.000.000/bulan selama 1 tahun + pelatihan soft skills (leadership, nation building, international exposure)',
        'deadline': '2026-06-10',
        'url': 'https://djarumbeasiswaplus.org'
    },
    {
        'id': 'B002',
        'nama_beasiswa': 'Beasiswa BCA Finance Peduli 2026',
        'penyelenggara': 'PT BCA Finance',
        'jenjang': 'S1,D4',
        'ipk_min': 3.30,
        'bidang': 'Semua Jurusan',
        'syarat': 'Semester 4-6, IPK min 3.30 (PTN) / 3.50 (PTS), aktif organisasi, tidak menerima beasiswa lain, SKTM dari kelurahan',
        'benefit': 'Rp3.500.000/semester + pelatihan online persiapan kerja',
        'deadline': '2026-06-21',
        'url': 'https://bcafinance.co.id'
    },
    {
        'id': 'B003',
        'nama_beasiswa': 'Beasiswa Etos ID 2026',
        'penyelenggara': 'Dompet Dhuafa & GREAT Edunesia',
        'jenjang': 'S1',
        'ipk_min': 0.00,
        'bidang': 'Semua Jurusan',
        'syarat': 'Mahasiswa baru PTN 2026 (jalur SNBP/SNBT), beragama Islam, keluarga prasejahtera, bersedia ikut pembinaan 8 semester, punya prestasi akademik/organisasi',
        'benefit': 'Bantuan UKT hingga 8 semester + uang saku Rp600.000/bulan (6 semester) + fasilitas asrama 1 tahun + program pengembangan diri',
        'deadline': '2026-06-10',
        'url': 'https://etosid.org'
    },
    {
        'id': 'B004',
        'nama_beasiswa': 'Beasiswa DataPrint 2026 Periode 2',
        'penyelenggara': 'DataPrint Indonesia',
        'jenjang': 'D3,D4,S1',
        'ipk_min': 0.00,
        'bidang': 'Semua Jurusan',
        'syarat': 'Mahasiswa aktif D3/D4/S1, aktif organisasi, tidak terlibat narkoba/kriminal, tidak menerima beasiswa swasta lain, memiliki kupon produk DataPrint, buat video essay min 3 menit',
        'benefit': 'Rp500.000 (satu kali)',
        'deadline': '2026-06-26',
        'url': 'https://beasiswadataprint.com'
    },
    {
        'id': 'B005',
        'nama_beasiswa': 'Beasiswa Glow and Lovely 2026',
        'penyelenggara': 'Glow and Lovely (Unilever)',
        'jenjang': 'S1',
        'ipk_min': 0.00,
        'bidang': 'Semua Jurusan',
        'syarat': 'Khusus perempuan, mahasiswa S1 aktif, WNI, punya motivasi kuat dan visi masa depan jelas',
        'benefit': 'Bantuan biaya kuliah + program mentoring dan pengembangan diri',
        'deadline': '2026-06-30',
        'url': 'https://kelasberbagicerah.com'
    }
]

In [ ]:
# ============================================================
# FUNGSI SCRAPING DARI INDBEASISWA.COM
# ============================================================

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'id-ID,id;q=0.9,en-US;q=0.8'
}

def scrape_indbeasiswa_listing(url, max_items=10):
    """
    Scrape daftar beasiswa dari indbeasiswa.com
    Return: list of dict {nama, url, deadline_raw}
    """
    results = []
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'lxml')

        # INDBeasiswa menggunakan <article> atau <h2> untuk daftar beasiswa
        articles = soup.select('article.post') or soup.select('h2.entry-title')

        for art in articles[:max_items]:
            title_tag = art.find('a') if art.name == 'article' else art.find('a')
            if title_tag:
                nama = title_tag.get_text(strip=True)
                link = title_tag.get('href', '')
                # Ekstrak deadline dari judul (pola: Deadline: DD Bulan YYYY)
                deadline_match = re.search(
                    r'Deadline[:\s]+([\d]+\s+\w+\s+\d{4})',
                    nama, re.IGNORECASE
                )
                deadline_raw = deadline_match.group(1) if deadline_match else 'N/A'
                results.append({
                    'nama': nama[:80],  # trim panjang judul
                    'url': link,
                    'deadline_raw': deadline_raw
                })
    except Exception as e:
        print(f'⚠️  Gagal scrape {url}: {e}')

    return results


def scrape_detail_beasiswa(url):
    """
    Scrape halaman detail beasiswa dari indbeasiswa.com
    Return: dict berisi info detail
    """
    detail = {
        'penyelenggara': 'N/A',
        'jenjang': 'N/A',
        'ipk_min': 0.0,
        'bidang': 'N/A',
        'syarat': 'N/A',
        'benefit': 'N/A',
        'deadline': 'N/A'
    }
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'lxml')

        # Ambil konten artikel utama
        content = soup.select_one('.entry-content') or soup.select_one('article')
        if not content:
            return detail

        full_text = content.get_text(' ', strip=True)

        # -- Penyelenggara --
        m = re.search(r'(?:Penyelenggara|Pemberi Beasiswa)[:\s]+([^\n.]{3,60})', full_text, re.I)
        if m:
            detail['penyelenggara'] = m.group(1).strip()

        # -- Jenjang --
        jenjang_found = re.findall(r'\b(S1|S2|S3|D3|D4)\b', full_text)
        if jenjang_found:
            detail['jenjang'] = ','.join(sorted(set(jenjang_found)))

        # -- IPK minimum --
        m = re.search(
            r'(?:IPK|IP Kumulatif)[^\d]*(\d[\.,]\d{1,2})',
            full_text, re.I
        )
        if m:
            detail['ipk_min'] = float(m.group(1).replace(',', '.'))

        # -- Bidang --
        if re.search(r'semua (jurusan|bidang|program studi)', full_text, re.I):
            detail['bidang'] = 'Semua Jurusan'
        else:
            m = re.search(r'(?:jurusan|bidang studi|program studi)[:\s]+([^\n.]{5,80})', full_text, re.I)
            if m:
                detail['bidang'] = m.group(1).strip()

        # -- Syarat (ambil kalimat yang mengandung kata kunci syarat) --
        syarat_keywords = ['semester', 'aktif', 'WNI', 'mahasiswa', 'tidak sedang']
        syarat_lines = []
        for sent in full_text.split('.'):
            if any(k.lower() in sent.lower() for k in syarat_keywords):
                cleaned = sent.strip()
                if 10 < len(cleaned) < 200:
                    syarat_lines.append(cleaned)
                if len(syarat_lines) >= 3:
                    break
        detail['syarat'] = '; '.join(syarat_lines) if syarat_lines else 'Lihat situs resmi'

        # -- Benefit --
        m = re.search(
            r'(?:benefit|cakupan|diberikan berupa|bantuan)[^\n]{0,20}:?([^\n.]{10,200})',
            full_text, re.I
        )
        if m:
            detail['benefit'] = m.group(1).strip()[:200]

        # -- Deadline --
        m = re.search(
            r'(?:deadline|batas waktu|ditutup)[^\d]*(\d{1,2}[\s/\-]\w+[\s/\-]\d{4})',
            full_text, re.I
        )
        if m:
            detail['deadline'] = m.group(1).strip()

    except Exception as e:
        print(f'⚠️  Gagal scrape detail {url}: {e}')

    return detail

In [ ]:
# ============================================================
# JALANKAN SCRAPING LISTING
# ============================================================

TARGET_URL = 'https://indbeasiswa.com/daftar-beasiswa-mahasiswa-s1/'

print(f'🔍 Scraping daftar beasiswa dari: {TARGET_URL}')
listing = scrape_indbeasiswa_listing(TARGET_URL, max_items=10)

print(f'\n📋 Ditemukan {len(listing)} item dari listing:')
for i, item in enumerate(listing, 1):
    print(f'  [{i}] {item["nama"][:70]}...')
    print(f'       Deadline: {item["deadline_raw"]} | URL: {item["url"][:60]}')

🔍 Scraping daftar beasiswa dari: https://indbeasiswa.com/daftar-beasiswa-mahasiswa-s1/

📋 Ditemukan 1 item dari listing:
  [1] Beasiswa Indonesia...
       Deadline: N/A | URL: https://indbeasiswa.com/author/beasiswa-indonesia-2/


In [ ]:
# ============================================================
# SCRAPE DETAIL UNTUK SETIAP BEASISWA YANG DITEMUKAN
# ============================================================

scraped_rows = []

if listing:
    print('🔄 Mulai scraping detail setiap beasiswa...\n')
    for i, item in enumerate(listing[:5]):  # Ambil 5 beasiswa saja
        print(f'[{i+1}/5] Scraping: {item["nama"][:60]}...')
        detail = scrape_detail_beasiswa(item['url'])

        row = {
            'id': f'S{str(i+1).zfill(3)}',  # ID dari scraping
            'nama_beasiswa': item['nama'],
            'penyelenggara': detail['penyelenggara'],
            'jenjang': detail['jenjang'],
            'ipk_min': detail['ipk_min'],
            'bidang': detail['bidang'],
            'syarat': detail['syarat'],
            'benefit': detail['benefit'],
            'deadline': detail['deadline'],
            'url': item['url']
        }
        scraped_rows.append(row)
        print(f'    Penyelenggara: {detail["penyelenggara"]} | IPK: {detail["ipk_min"]}')
        time.sleep(2)  # Jeda antar request agar tidak di-block

    print(f'\n Scraping selesai: {len(scraped_rows)} beasiswa berhasil di-scrape')
else:
    print('  Tidak ada data dari scraping listing. Akan menggunakan dataset manual saja.')

🔄 Mulai scraping detail setiap beasiswa...

[1/5] Scraping: Beasiswa Indonesia...
    Penyelenggara: N/A | IPK: 0.0

 Scraping selesai: 1 beasiswa berhasil di-scrape


In [ ]:
# ============================================================
# BUAT DATAFRAME DARI DATA MANUAL (SUDAH TERVERIFIKASI)
# ============================================================

df_manual = pd.DataFrame(data_beasiswa)

print('=== Dataset Manual (5 Beasiswa Aktif Juni 2026) ===')
print(df_manual.to_string(index=False))

=== Dataset Manual (5 Beasiswa Aktif Juni 2026) ===
  id                     nama_beasiswa                  penyelenggara  jenjang  ipk_min        bidang                                                                                                                                                                          syarat                                                                                                                       benefit   deadline                            url
B001         Djarum Beasiswa Plus 2026              Djarum Foundation    S1,D4      3.0 Semua Jurusan                                                                    Semester 4, IPK min 3.00 di sem 3, aktif organisasi, tidak menerima beasiswa lain, kuliah di PT mitra Djarum                Rp1.000.000/bulan selama 1 tahun + pelatihan soft skills (leadership, nation building, international exposure) 2026-06-10 https://djarumbeasiswaplus.org
B002  Beasiswa BCA Finance Peduli 2026                 P

## 🗄️ Data Beasiswa (Database Mock)
Representasi database beasiswa yang bisa di-scrape dari sumber resmi

In [ ]:
# ==========================================
# SIMPAN CSV
# ==========================================
# ============================================================
# GABUNGKAN DENGAN HASIL SCRAPING (JIKA ADA)
# ============================================================

if scraped_rows:
    df_scraped = pd.DataFrame(scraped_rows)

    # Hindari duplikat berdasarkan nama beasiswa
    all_names_manual = set(df_manual['nama_beasiswa'].str.lower().str.strip())
    df_scraped_new = df_scraped[
        ~df_scraped['nama_beasiswa'].str.lower().str.strip().isin(all_names_manual)
    ]

    df_final = pd.concat([df_manual, df_scraped_new], ignore_index=True)
    print(f'✅ Digabungkan: {len(df_manual)} manual + {len(df_scraped_new)} scraping = {len(df_final)} total')
else:
    df_final = df_manual.copy()
    print('✅ Menggunakan dataset manual saja.')

# Reset ID agar urut
df_final['id'] = ['B' + str(i+1).zfill(3) for i in range(len(df_final))]

print(f'\n Total beasiswa dalam dataset: {len(df_final)}')
df_final.head()
df = pd.DataFrame(results)

df.to_csv(
    "scholarships_scraped.csv",
    index=False
)

print("\nSelesai")
print(df.head())

from google.colab import files
files.download("scholarships_scraped.csv")

✅ Digabungkan: 5 manual + 1 scraping = 6 total

 Total beasiswa dalam dataset: 6

Selesai
                         nama      penyelenggara  \
0                        LPDP               LPDP   
1        Djarum Beasiswa Plus  Djarum Foundation   
2  Bank Indonesia Scholarship     Bank Indonesia   
3   Tanoto Foundation TELADAN  Tanoto Foundation   
4           Beasiswa Unggulan        Kemendikbud   

                                        url  \
0               https://lpdp.kemenkeu.go.id   
1            https://djarumbeasiswaplus.org   
2                      https://www.bi.go.id   
3          https://www.tanotofoundation.org   
4  https://beasiswaunggulan.kemdikbud.go.id   

                                         syarat_text  
0  LPDP - Lembaga Pengelola Dana Pendidikan Gedun...  
1  Djarum Beasiswa Plus | Program Beasiswa Presta...  
2  Bank Indonesia Turn on more accessible mode Tu...  
3  Home - Tanoto Foundation About Us Our Works Ed...  
4                                      

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# CLEANING & NORMALISASI
# ============================================================

def normalize_jenjang(jenjang_str):
    """Normalisasi kolom jenjang ke format standar."""
    valid = ['D3', 'D4', 'S1', 'S2', 'S3']
    found = re.findall(r'\b(D3|D4|S1|S2|S3)\b', str(jenjang_str).upper())
    return ','.join(sorted(set(found))) if found else jenjang_str

def normalize_deadline(deadline_str):
    """Konversi berbagai format tanggal ke YYYY-MM-DD."""
    bulan_map = {
        'januari': '01', 'februari': '02', 'maret': '03',
        'april': '04', 'mei': '05', 'juni': '06',
        'juli': '07', 'agustus': '08', 'september': '09',
        'oktober': '10', 'november': '11', 'desember': '12',
        'january': '01', 'february': '02', 'march': '03',
        'april': '04', 'may': '05', 'june': '06',
        'july': '07', 'august': '08', 'september': '09',
        'october': '10', 'november': '11', 'december': '12'
    }
    s = str(deadline_str).strip()

    # Sudah format YYYY-MM-DD
    if re.match(r'^\d{4}-\d{2}-\d{2}$', s):
        return s

    # Format: "21 Juni 2026" atau "21 June 2026"
    m = re.match(r'(\d{1,2})\s+(\w+)\s+(\d{4})', s, re.I)
    if m:
        dd = m.group(1).zfill(2)
        bln = bulan_map.get(m.group(2).lower(), '00')
        yy = m.group(3)
        if bln != '00':
            return f'{yy}-{bln}-{dd}'

    return s  # Kembalikan apa adanya jika tidak bisa diparse


# Apply normalisasi
df_clean = df_final.copy()
df_clean['jenjang']   = df_clean['jenjang'].apply(normalize_jenjang)
df_clean['deadline']  = df_clean['deadline'].apply(normalize_deadline)
df_clean['ipk_min']   = pd.to_numeric(df_clean['ipk_min'], errors='coerce').fillna(0.0).round(2)
df_clean['bidang']    = df_clean['bidang'].str.strip()
df_clean['nama_beasiswa'] = df_clean['nama_beasiswa'].str.strip()

# Pastikan urutan kolom sesuai format target
COLS = ['id', 'nama_beasiswa', 'penyelenggara', 'jenjang', 'ipk_min',
        'bidang', 'syarat', 'benefit', 'deadline', 'url']
df_clean = df_clean[COLS]

print('✅ Cleaning selesai!')
print(f'   Shape: {df_clean.shape}')
print(f'   Missing values:\n{df_clean.isnull().sum()}')

✅ Cleaning selesai!
   Shape: (6, 10)
   Missing values:
id               0
nama_beasiswa    0
penyelenggara    0
jenjang          0
ipk_min          0
bidang           0
syarat           0
benefit          0
deadline         0
url              0
dtype: int64


In [ ]:
# ============================================================
# SIMPAN DATASET KE CSV
# ============================================================

OUTPUT_FILE = 'dataset_beasiswa.csv'

df_clean.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig', sep=';')

print(f'✅ Dataset berhasil disimpan ke: {OUTPUT_FILE}')
print(f'   Total baris: {len(df_clean)}')
print(f'   Kolom: {list(df_clean.columns)}')

# Download otomatis di Google Colab
try:
    from google.colab import files
    files.download(OUTPUT_FILE)
    print('\n📥 File CSV sedang didownload...')
except ImportError:
    print('\nℹ️  Jalankan di Google Colab untuk auto-download.')

✅ Dataset berhasil disimpan ke: dataset_beasiswa.csv
   Total baris: 6
   Kolom: ['id', 'nama_beasiswa', 'penyelenggara', 'jenjang', 'ipk_min', 'bidang', 'syarat', 'benefit', 'deadline', 'url']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


📥 File CSV sedang didownload...


In [ ]:
import pandas as pd

def load_scholarships_from_csv(csv_path="dataset_beasiswa.csv"):
    df = pd.read_csv(csv_path)

    # Buat kolom matching_text: gabungan teks untuk BM25 + Sentence-BERT
    df["matching_text"] = (
        df["nama_beasiswa"].fillna("") + " " +
        df["jenjang"].fillna("") + " " +
        df["bidang"].fillna("") + " " +
        df["syarat"].fillna("") + " " +
        df["benefit"].fillna("")
    )

    # Convert ke format yang dipakai model teman: list of dict
    scholarships = df[["id", "nama_beasiswa", "matching_text"]].rename(
        columns={"nama_beasiswa": "nama", "matching_text": "syarat"}
    ).to_dict(orient="records")

    documents = [s["syarat"] for s in scholarships]

    print(f"Loaded {len(scholarships)} beasiswa dari CSV")
    return scholarships, documents

In [ ]:
# Library sudah diinstall di cell pertama ✅

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 7.7 MB/s eta 0:00:00


In [ ]:
# Import sudah dilakukan di cell pertama ✅

In [ ]:
# ==========================================
# 1. PIPELINE PARSING PDF (MENDUKUNG ZIP)
# ==========================================
import os
import zipfile
import io
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path, zip_path='CV-20260624T120554Z-3-001.zip'):
    """Fungsi untuk mengekstrak teks dari file PDF CV user (bisa dari lokal atau dari dalam ZIP)"""
    print(f'[*] Membaca file PDF: {pdf_path}...')
    
    # 1. Cek apakah file ada secara lokal
    if os.path.exists(pdf_path):
        try:
            reader = PdfReader(pdf_path)
            text = ''
            for page in reader.pages:
                extracted = page.extract_text()
                if extracted:
                    text += extracted + ' '
            print('[*] Ekstraksi teks selesai (Lokal).\n')
            return text.strip()
        except Exception as e:
            print(f'[!] Terjadi kesalahan saat membaca PDF lokal: {e}')
            return ''
            
    # 2. Cek apakah ada di dalam file ZIP
    if os.path.exists(zip_path):
        try:
            with zipfile.ZipFile(zip_path, 'r') as zf:
                normalized_path = pdf_path.replace('\\', '/')
                found_name = None
                # Cari file yang namanya cocok (baik path lengkap atau hanya nama file)
                for name in zf.namelist():
                    if name.replace('\\', '/') == normalized_path or os.path.basename(name).lower() == os.path.basename(pdf_path).lower():
                        found_name = name
                        break
                
                if found_name:
                    print(f"[*] Menemukan file '{found_name}' di dalam ZIP '{zip_path}'. Membaca langsung...")
                    with zf.open(found_name) as pdf_file:
                        # Gunakan BytesIO karena PdfReader butuh seekable stream
                        pdf_data = io.BytesIO(pdf_file.read())
                        reader = PdfReader(pdf_data)
                        text = ''
                        for page in reader.pages:
                            extracted = page.extract_text()
                            if extracted:
                                text += extracted + ' '
                        print('[*] Ekstraksi teks selesai (dari ZIP).\n')
                        return text.strip()
                else:
                    print(f"[!] File '{pdf_path}' tidak ditemukan secara lokal maupun di dalam ZIP '{zip_path}'.")
                    # Rekomendasikan nama file yang mirip
                    search_query = os.path.basename(pdf_path).lower().replace('cv_', '').replace('.pdf', '').strip()
                    matches = [name for name in zf.namelist() if search_query and search_query in name.lower()]
                    if matches:
                        print("Apakah yang dimaksud adalah salah satu dari file berikut di dalam ZIP?")
                        for m in matches:
                            print(f"  - {m}")
                    return ''
        except Exception as e:
            print(f'[!] Terjadi kesalahan saat membaca file ZIP: {e}')
            return ''
            
    print(f"[!] File PDF '{pdf_path}' tidak ditemukan, dan file ZIP '{zip_path}' juga tidak ditemukan.")
    return ''


---
## 🧩 Step 3 — Feature Extraction dari CV
> **BARU** — Mengekstrak IPK, Organisasi, dan Prestasi dari teks CV menggunakan regex parser.

| Fitur | Cara deteksi | Skala skor |
|-------|-------------|------------|
| IPK | Regex: `ipk 3.85`, `GPA: 3.85`, `3.85/4.00` | Nilai aktual (0–4) |
| Organisasi | Keyword jabatan (Ketua/Pengurus/Anggota) | 0, 0.6, 0.8, 1.0 |
| Prestasi | Keyword level (Internasional/Nasional/dll) | 0, 0.35, 0.6, 0.8, 1.0 |

In [ ]:
# ============================================================
# STEP 3 — FEATURE EXTRACTION DARI TEKS CV
# ============================================================

def extract_features_from_cv(cv_text):
    """
    Ekstrak fitur terstruktur dari teks CV.

    Returns dict:
        ipk              : float | None  (None = tidak terdeteksi di CV)
        ipk_detected     : bool
        org_score        : float 0.0-1.0
        org_level        : str   label jabatan
        achievement_score: float 0.0-1.0
        achievement_level: str   label tingkat prestasi
        jurusan          : str
    """
    tl = cv_text.lower()

    # ── IPK Parser ──────────────────────────────────────────────
    # Coba berbagai format: 'IPK 3.85', 'GPA: 3.75', '3.85/4.00', dll
    ipk = None
    for pat in [
        r'ipk\s*[:\-]?\s*([3-4][.,]\d{2})',
        r'gpa\s*[:\-]?\s*([3-4][.,]\d{2})',
        r'indeks prestasi kumulatif\s*[:\-]?\s*([3-4][.,]\d{2})',
        r'([3-4][.,]\d{2})\s*/\s*4[.,]?00',   # format 3.85/4.00
        r'([3-4][.,]\d{2})\s*/\s*4(?!\d)',    # format 3.85/4
    ]:
        m = re.search(pat, tl)
        if m:
            try:
                val = float(m.group(1).replace(',', '.'))
                if 0.0 < val <= 4.0:
                    ipk = val
                    break
            except ValueError:
                pass

    # ── Jurusan Parser ──────────────────────────────────────────
    jurusan = ''
    for pat in [
        r'(teknologi sains data|data science|informatika|teknik informatika|'
        r'sistem informasi|nanotechnology|teknik elektro|manajemen|ekonomi|'
        r'statistika|matematika|fisika|kimia|biologi|kedokteran|hukum)',
        r'(?:program studi|prodi|jurusan)\s*[:\-]?\s*([\w\s]+?)(?:\n|\||,)',
    ]:
        m = re.search(pat, tl)
        if m:
            jurusan = m.group(1).strip()[:40].title()
            break

    # ── Organisasi Scorer ────────────────────────────────────────
    # Skor diambil dari jabatan TERTINGGI yang ditemukan di CV
    org_score, org_level = 0.0, 'tidak ada'
    if re.search(r'\b(ketua umum|chairman|president|head of|kepala divisi|'
                 r'ketua himpunan|ketua bem|ketua ukm|ketua panitia|ketua pelaksana)\b', tl):
        org_score, org_level = 1.0, 'ketua'
    elif re.search(r'\bketua\b', tl):
        org_score, org_level = 1.0, 'ketua'
    elif re.search(r'\b(wakil ketua|vice|sekretaris jenderal|sekjen|'
                   r'bendahara umum|koordinator|pengurus harian)\b', tl):
        org_score, org_level = 0.8, 'pengurus senior'
    elif re.search(r'\b(sekretaris|bendahara|pengurus|staff ahli|'
                   r'kepala bidang|kepala departemen)\b', tl):
        org_score, org_level = 0.8, 'pengurus'
    elif re.search(r'\b(anggota|member|relawan|volunteer|staff divisi|divisi)\b', tl):
        org_score, org_level = 0.6, 'anggota'
    elif re.search(r'\b(himpunan|bem|ukm|osis|komunitas|organisasi|panitia)\b', tl):
        org_score, org_level = 0.6, 'partisipan'

    # ── Prestasi Scorer ──────────────────────────────────────────
    # Skor diambil dari tingkat prestasi TERTINGGI yang ditemukan
    achievement_score, achievement_level = 0.0, 'tidak ada'
    has_award = bool(re.search(
        r'\b(juara|winner|finalis|semi.?final|penghargaan|award|'
        r'medali|lomba|kompetisi|olimpiade|hackathon)\b', tl
    ))
    if has_award:
        if re.search(r'\b(internasional|international|world|global|asean|asia)\b', tl):
            achievement_score, achievement_level = 1.0, 'internasional'
        elif re.search(r'\b(nasional|national|tingkat nasional)\b', tl):
            achievement_score, achievement_level = 0.8, 'nasional'
        elif re.search(r'\b(provinsi|regional|daerah|kota|kabupaten)\b', tl):
            achievement_score, achievement_level = 0.6, 'provinsi'
        else:
            achievement_score, achievement_level = 0.35, 'kampus/lainnya'

    features = {
        'ipk'               : ipk,
        'ipk_detected'      : ipk is not None,
        'org_score'         : org_score,
        'org_level'         : org_level,
        'achievement_score' : achievement_score,
        'achievement_level' : achievement_level,
        'jurusan'           : jurusan,
    }

    print('[Feature Extraction]')
    print(f'  IPK         : {ipk if ipk else "tidak terdeteksi"}')
    print(f'  Jurusan     : {jurusan or "tidak terdeteksi"}')
    print(f'  Organisasi  : {org_level} → skor {org_score}')
    print(f'  Prestasi    : {achievement_level} → skor {achievement_score}')
    return features


---
## ⚖️ Step 4 — WSM/MCDM Engine
> **BARU** — Empat fungsi inti:

| Fungsi | Peran |
|--------|-------|
| `generate_dynamic_weights()` | Bobot otomatis dari requirement beasiswa |
| `calculate_ipk_score()` | Gap score IPK user vs IPK minimum |
| `calculate_major_score()` | Kesesuaian jurusan CV vs bidang beasiswa |
| `compute_wsm_score()` | Hitung WSM = Σ w_i × x_i |

In [ ]:
# ============================================================
# STEP 4 — WSM / MCDM ENGINE (LENGKAP)
# Implementasi 1: Dynamic Weight Generation
# Implementasi 2: IPK Gap Score
# Implementasi 3: Jurusan Matching Score
# ============================================================

# ── IMPLEMENTASI 1: Dynamic Weight Generation ────────────────
def generate_dynamic_weights(requirements):
    """
    Membentuk bobot WSM otomatis dari requirement beasiswa.

    Prinsip MCDM:
    - Setiap kriteria yang muncul di syarat beasiswa mendapat bobot
    - Semakin banyak kriteria → bobot dibagi merata (equal weight)
    - Bobot total selalu = 1.0

    Contoh:
    requirements = {'ipk': True, 'organisasi': True, 'prestasi': True}
    → {'ipk': 0.333, 'organisasi': 0.333, 'prestasi': 0.333}

    requirements = {'ipk': True, 'organisasi': True}
    → {'ipk': 0.5, 'organisasi': 0.5}
    """
    active_features = [key for key, value in requirements.items() if value]

    if len(active_features) == 0:
        # Tidak ada syarat spesifik → semua kriteria dengan bobot sama
        active_features = ['ipk', 'organisasi', 'prestasi']

    weight = round(1.0 / len(active_features), 4)
    return {feature: weight for feature in active_features}


# ── IMPLEMENTASI 2: IPK Gap Score ────────────────────────────
def calculate_ipk_score(user_ipk, ipk_min):
    """
    Mengukur seberapa baik IPK user dibanding syarat minimum.

    Formula: score = min(user_ipk / ipk_min, 1.0)

    Contoh:
    user_ipk=3.85, ipk_min=3.50 → 3.85/3.50 = 1.0  (memenuhi, di-cap 1.0)
    user_ipk=3.20, ipk_min=3.50 → 3.20/3.50 = 0.91 (mendekati tapi kurang)
    user_ipk=0,    ipk_min=3.50 → 0          (IPK tidak terdeteksi di CV)
    """
    if not user_ipk or user_ipk <= 0:
        return 0.0  # IPK tidak terdeteksi

    if ipk_min <= 0:
        # Beasiswa tanpa IPK minimum → normalisasi ke skala 4.0
        return round(min(user_ipk / 4.0, 1.0), 4)

    return round(min(user_ipk / ipk_min, 1.0), 4)


# ── IMPLEMENTASI 3: Jurusan Matching Score ────────────────────
def calculate_major_score(user_major, scholarship_major):
    """
    Mengukur kesesuaian jurusan CV dengan bidang beasiswa.

    Aturan:
    - 'Semua Jurusan' → selalu 1.0 (tidak ada pembatasan)
    - Jurusan user ada dalam bidang beasiswa → 1.0 (exact/partial match)
    - Tidak ada kecocokan → 0.3 (tetap diberi skor kecil,
      karena beasiswa bisa lintas bidang)
    """
    if not user_major:
        return 0.5  # Jurusan tidak terdeteksi → skor netral

    scholar_lower = str(scholarship_major).lower()
    user_lower    = str(user_major).lower()

    # Beasiswa terbuka semua jurusan
    if 'semua jurusan' in scholar_lower or 'all' in scholar_lower:
        return 1.0

    # Cek kecocokan kata kunci jurusan
    # Pecah jurusan user jadi kata-kata, cek satu per satu
    user_words = re.sub(r'[^a-z\s]', '', user_lower).split()
    for word in user_words:
        if len(word) > 3 and word in scholar_lower:
            return 1.0

    # Cek sebaliknya: kata dari bidang beasiswa ada di jurusan user
    scholar_words = re.sub(r'[^a-z\s]', '', scholar_lower).split()
    for word in scholar_words:
        if len(word) > 3 and word in user_lower:
            return 1.0

    return 0.3  # Tidak ada kecocokan


# ── extract_scholarship_requirements (tidak berubah) ─────────
def extract_scholarship_requirements(syarat_text):
    """
    Deteksi kriteria yang disyaratkan dari kolom 'syarat' beasiswa.
    Returns dict: {ipk: bool, organisasi: bool, prestasi: bool}
    """
    tl = str(syarat_text).lower()
    return {
        'ipk'        : bool(re.search(
            r'\b(ipk|ip kumulatif|indeks prestasi|nilai akademik)\b', tl)),
        'organisasi' : bool(re.search(
            r'\b(organisasi|aktif|himpunan|bem|ukm|kepanitiaan|kegiatan mahasiswa)\b', tl)),
        'prestasi'   : bool(re.search(
            r'\b(prestasi|penghargaan|juara|award|kompetisi|lomba|olimpiade)\b', tl)),
    }


# ── compute_wsm_score: sekarang pakai calculate_ipk_score ─────
def compute_wsm_score(user_features, scholarship_row, weights):
    """
    Hitung Weighted Sum Model score.
    Formula: WSM = Σ w_i × x_i

    Returns: (wsm_score: float, components: dict)
    """
    components = {}

    if 'ipk' in weights:
        ipk_min = float(scholarship_row.get('ipk_min', 0) or 0)
        components['ipk'] = calculate_ipk_score(user_features['ipk'], ipk_min)

    if 'organisasi' in weights:
        components['organisasi'] = user_features['org_score']

    if 'prestasi' in weights:
        components['prestasi'] = user_features['achievement_score']

    wsm = sum(weights[k] * components.get(k, 0.0) for k in weights)
    return round(wsm, 4), components


# ── Preview: requirements + bobot dinamis tiap beasiswa ───────
print('Analisis syarat, bobot dinamis, & major score per beasiswa:')
print(f'\n{"Beasiswa":<35} {"Syarat":<25} {"Bobot Dinamis"}')
print('-' * 85)
for _, row in df_clean.iterrows():
    req   = extract_scholarship_requirements(row['syarat'])
    aktif = [k for k, v in req.items() if v] or ['umum']
    w     = generate_dynamic_weights(req)
    print(f"{row['nama_beasiswa'][:34]:<35} {str(aktif):<25} {w}")


---
## 🔍 Step 5 — Semantic Matching (BM25 + Sentence Transformer)
> **EXISTING** — Fungsi `run_hybrid_realistic` di atas dipertahankan.
> Di Step 6 kita wrap menjadi fungsi yang mengembalikan skor numerik,
> bukan hanya print, supaya bisa digabungkan dengan WSM.

In [ ]:
# ============================================================
# STEP 5 — SEMANTIC MATCHING (refactor dari run_hybrid_realistic)
# Fungsi baru yang RETURN skor, bukan hanya print
# ============================================================

def build_semantic_index(documents):
    """
    Bangun BM25 index dan pre-compute Sentence Transformer embeddings.
    Dipanggil SEKALI sebelum loop user agar efisien.

    Returns: bm25, doc_embeddings, st_model
    """
    # BM25 lexical index
    tokenized = [doc.lower().split() for doc in documents]
    bm25 = BM25Okapi(tokenized)

    # Sentence Transformer (model kecil, cepat)
    print('[*] Loading Sentence Transformer model...')
    model = SentenceTransformer('all-MiniLM-L6-v2')
    print('[*] Encoding dokumen beasiswa...')
    doc_embeddings = model.encode(documents, show_progress_bar=True)
    print('✅ Semantic index siap\n')
    return bm25, doc_embeddings, model


def compute_semantic_scores(cv_text, bm25, doc_embeddings, st_model):
    """
    Hitung semantic score antara satu CV dan semua beasiswa.

    Fusion: 0.3 × BM25_norm + 0.7 × cosine_norm

    Returns: np.array shape (n_beasiswa,)
    """
    # BM25
    bm25_raw  = np.array(bm25.get_scores(cv_text.lower().split()))
    bm25_norm = (bm25_raw - bm25_raw.min()) / (bm25_raw.max() - bm25_raw.min() + 1e-9)

    # Cosine similarity
    cv_emb    = st_model.encode([cv_text])
    sem_raw   = cosine_similarity(cv_emb, doc_embeddings)[0]
    sem_norm  = (sem_raw - sem_raw.min()) / (sem_raw.max() - sem_raw.min() + 1e-9)

    # Sama seperti run_hybrid_realistic: 30% lexical + 70% semantic
    return 0.3 * bm25_norm + 0.7 * sem_norm


# Build index sekali (model di-load dan beasiswa di-encode)
scholarships, documents = load_scholarships_from_csv('dataset_beasiswa.csv')
bm25_index, doc_embeddings, st_model = build_semantic_index(documents)


---
## 🔀 Step 6 — Hybrid Fusion: Semantic + WSM + Major Matching
> **BARU** — Formula final dengan 3 komponen:

```
FinalScore = 0.50 × semantic_score
           + 0.30 × wsm_score
           + 0.20 × major_score
```

| Komponen | Bobot | Penjelasan |
|----------|-------|------------|
| Semantic Matching | 50% | Kemiripan konten CV vs beasiswa (BM25 + SBERT) |
| WSM Feature Score | 30% | IPK gap score + organisasi + prestasi |
| Major Matching | 20% | Kesesuaian jurusan CV dengan bidang beasiswa |

Setiap rekomendasi dilengkapi **penjelasan (explanation)** → nilai tambah saat demo ke dosen.

In [ ]:
# ============================================================
# STEP 6 — HYBRID RECOMMENDATION (Semantic + WSM + Major)
# Implementasi 4: Explainable Recommendation
# Implementasi 5: Final Hybrid Score 3-komponen
# ============================================================

# ── IMPLEMENTASI 4: Explanation Engine ───────────────────────
def generate_explanation(user_features, requirements, scholarship_row):
    """
    Hasilkan penjelasan natural language mengapa beasiswa ini direkomendasikan.

    Returns: list of str (kalimat penjelasan)
    """
    explanations = []
    ipk_min = float(scholarship_row.get('ipk_min', 0) or 0)

    # ── IPK ──────────────────────────────────────────────────
    if requirements.get('ipk'):
        if user_features['ipk_detected']:
            user_ipk = user_features['ipk']
            if ipk_min > 0 and user_ipk >= ipk_min:
                explanations.append(
                    f'✓ IPK {user_ipk:.2f} memenuhi syarat minimum {ipk_min:.2f}'
                )
            elif ipk_min > 0 and user_ipk < ipk_min:
                explanations.append(
                    f'⚠ IPK {user_ipk:.2f} belum memenuhi syarat minimum {ipk_min:.2f} '
                    f'(gap: {ipk_min - user_ipk:.2f})'
                )
            else:
                explanations.append(f'✓ IPK {user_ipk:.2f} (tidak ada batas minimum)')
        else:
            explanations.append('– IPK tidak tercantum di CV')

    # ── Organisasi ───────────────────────────────────────────
    if requirements.get('organisasi'):
        lvl = user_features['org_level']
        if lvl != 'tidak ada':
            explanations.append(
                f'✓ Memiliki pengalaman organisasi sebagai {lvl} '
                f'(skor: {user_features["org_score"]})'
            )
        else:
            explanations.append('⚠ Tidak ditemukan pengalaman organisasi di CV')

    # ── Prestasi ─────────────────────────────────────────────
    if requirements.get('prestasi'):
        lvl = user_features['achievement_level']
        if lvl != 'tidak ada':
            explanations.append(
                f'✓ Memiliki prestasi tingkat {lvl} '
                f'(skor: {user_features["achievement_score"]})'
            )
        else:
            explanations.append('⚠ Tidak ditemukan prestasi di CV')

    # ── Jurusan ───────────────────────────────────────────────
    bidang = scholarship_row.get('bidang', '')
    jurusan = user_features.get('jurusan', '')
    major_sc = calculate_major_score(jurusan, bidang)
    if major_sc == 1.0:
        if 'semua jurusan' in str(bidang).lower():
            explanations.append(f'✓ Beasiswa terbuka untuk semua jurusan')
        else:
            explanations.append(
                f'✓ Jurusan "{jurusan or "tidak terdeteksi"}" '
                f'sesuai bidang beasiswa "{bidang}"'
            )
    else:
        explanations.append(
            f'– Jurusan "{jurusan or "tidak terdeteksi"}" '
            f'berbeda dengan bidang "{bidang}"'
        )

    return explanations


# ── IMPLEMENTASI 5: run_hybrid_wsm dengan formula 3 komponen ─
def run_hybrid_wsm(cv_text, df_beasiswa, bm25_index, doc_embeddings,
                   st_model, scholarships,
                   w_semantic=0.50, w_wsm=0.30, w_major=0.20):
    """
    Pipeline Hybrid Recommendation — 3 komponen skor.

    Formula:
        FinalScore = w_semantic × semantic_score
                   + w_wsm     × wsm_score
                   + w_major   × major_score

    Parameters
    ----------
    cv_text     : str     teks CV user
    df_beasiswa : DataFrame  dataset beasiswa (df_clean)
    w_semantic  : float   bobot semantic  (default 0.50)
    w_wsm       : float   bobot WSM       (default 0.30)
    w_major     : float   bobot jurusan   (default 0.20)

    Returns
    -------
    list of dict diurutkan final_score DESC
    """
    assert abs(w_semantic + w_wsm + w_major - 1.0) < 1e-6, \
        'Bobot harus berjumlah 1.0'

    print('=' * 62)
    print('HYBRID RECOMMENDATION — SEMANTIC + WSM + MAJOR MATCHING')
    print(f'  Bobot Semantic  : {w_semantic} (BM25 + Sentence-BERT)')
    print(f'  Bobot WSM       : {w_wsm}  (IPK + Organisasi + Prestasi)')
    print(f'  Bobot Jurusan   : {w_major}  (Major Matching)')
    print('=' * 62)

    # ── STEP 3: Ekstrak fitur user ────────────────────────────
    print('\n[1/3] Feature Extraction dari CV...')
    features = extract_features_from_cv(cv_text)

    # ── STEP 5: Semantic scores ───────────────────────────────
    print('\n[2/3] Menghitung Semantic Scores...')
    sem_scores = compute_semantic_scores(
        cv_text, bm25_index, doc_embeddings, st_model
    )

    # ── STEP 4 + FUSION per beasiswa ─────────────────────────
    print('[3/3] Menghitung WSM + Major + Hybrid Fusion...')
    results = []

    for i, (_, row) in enumerate(df_beasiswa.iterrows()):

        # Deteksi requirement beasiswa
        req = extract_scholarship_requirements(row['syarat'])

        # Bobot dinamis dari requirement (IMPLEMENTASI 1)
        weights = generate_dynamic_weights(req)

        # WSM score (pakai calculate_ipk_score di dalamnya)
        wsm_score, components = compute_wsm_score(features, row, weights)

        # Major matching score (IMPLEMENTASI 3)
        major_score = calculate_major_score(
            features.get('jurusan', ''),
            row.get('bidang', '')
        )

        # Final Score — formula 3 komponen (IMPLEMENTASI 5)
        final = round(
            w_semantic * float(sem_scores[i]) +
            w_wsm      * wsm_score            +
            w_major    * major_score,
            4
        )

        # Explanation (IMPLEMENTASI 4)
        explanation = generate_explanation(features, req, row)

        results.append({
            'id_beasiswa'      : row['id'],
            'nama_beasiswa'    : row['nama_beasiswa'],
            'penyelenggara'    : row['penyelenggara'],
            'deadline'         : row['deadline'],
            'ipk_min'          : row['ipk_min'],
            'bidang'           : row['bidang'],
            'url'              : row.get('url', ''),
            'final_score'      : final,
            'semantic_score'   : round(float(sem_scores[i]), 4),
            'wsm_score'        : wsm_score,
            'major_score'      : major_score,
            'wsm_weights'      : weights,
            'wsm_components'   : components,
            'req_detected'     : [k for k, v in req.items() if v] or ['umum'],
            'explanation'      : explanation,
        })

    # ── STEP 7: Ranking ───────────────────────────────────────
    results = sorted(results, key=lambda x: x['final_score'], reverse=True)

    # Tabel ranking ringkas
    print('\n' + '=' * 72)
    print('HASIL RANKING BEASISWA')
    print('=' * 72)
    print(f'{"Rank":<5} {"Beasiswa":<36} {"Final":>7} {"Sem":>6} {"WSM":>6} {"Major":>6}')
    print('-' * 72)
    for rank, r in enumerate(results, 1):
        print(f"{rank:<5} {r['nama_beasiswa'][:35]:<36} "
              f"{r['final_score']:>7.4f} {r['semantic_score']:>6.4f} "
              f"{r['wsm_score']:>6.4f} {r['major_score']:>6.2f}")

    # Detail + explanation top-N
    print('\n─── DETAIL & PENJELASAN TOP REKOMENDASI ───')
    for rank, r in enumerate(results, 1):
        print(f'\n{'═'*55}')
        print(f'  #{rank}  {r["nama_beasiswa"]}')
        print(f'{'═'*55}')
        print(f'  Penyelenggara  : {r["penyelenggara"]}')
        print(f'  Deadline       : {r["deadline"]}  |  IPK Min: {r["ipk_min"]}')
        print(f'  Bidang         : {r["bidang"]}')
        print(f'  URL            : {r["url"]}')
        print(f'  ─── Skor ───')
        print(f'  Final Score    : {r["final_score"]:.4f}')
        print(f'    {w_semantic} × {r["semantic_score"]:.4f} (semantic)'
              f'  +  {w_wsm} × {r["wsm_score"]:.4f} (WSM)'
              f'  +  {w_major} × {r["major_score"]:.2f} (major)')
        print(f'  Bobot Dinamis  : {r["wsm_weights"]}')
        print(f'  Komponen WSM   : {r["wsm_components"]}')
        print(f'  ─── Alasan Rekomendasi ───')
        for line in r['explanation']:
            print(f'    {line}')

    return results


---
## 🚀 Eksekusi Utama — Hybrid Recommendation
**Langkah penggunaan:**
1. Upload file PDF CV ke Google Colab (klik ikon folder di sidebar kiri)
2. Ganti `cv_filename` dengan nama file PDF kamu
3. Jalankan cell ini

Parameter yang bisa diubah:
- `ALPHA` — bobot semantic (0.4 = 40% semantic, 60% WSM)
- `TOP_N` — jumlah rekomendasi yang ditampilkan

In [ ]:
# ============================================================
# EKSEKUSI UTAMA — SISTEM REKOMENDASI BEASISWA HYBRID
# ============================================================

# ── Konfigurasi ───────────────────────────────────────────────
cv_filename = '/content/CV_Salwa Dewi Aqiilah.pdf'  # ← GANTI dengan nama file CV kamu

# Bobot 3 komponen — total harus = 1.0
W_SEMANTIC = 0.50   # Bobot semantic matching
W_WSM      = 0.30   # Bobot WSM (IPK + Organisasi + Prestasi)
W_MAJOR    = 0.20   # Bobot kesesuaian jurusan

TOP_N = 3           # Jumlah rekomendasi teratas yang di-highlight

print('=' * 55)
print('SISTEM REKOMENDASI BEASISWA — HYBRID (SEMANTIC + WSM)')
print('Kelompok G — Machine Learning SD-A2')
print('=' * 55)

# ── 1. Ekstrak teks CV ────────────────────────────────────────
cv_text = extract_text_from_pdf(cv_filename)

if not cv_text:
    print('[!] Gagal membaca CV. Pastikan file ada dan path benar.')
else:
    print(f'[✓] CV terbaca: {len(cv_text)} karakter')
    print(f'    Cuplikan  : {cv_text[:200]}\n')

    # ── 2. Jalankan Hybrid Recommendation ────────────────────
    hasil = run_hybrid_wsm(
        cv_text,
        df_clean,
        bm25_index,
        doc_embeddings,
        st_model,
        scholarships,
        w_semantic = W_SEMANTIC,
        w_wsm      = W_WSM,
        w_major    = W_MAJOR,
    )

    # ── 3. Tampilkan tabel hasil ──────────────────────────────
    df_hasil = pd.DataFrame(hasil)[[
        'nama_beasiswa', 'penyelenggara', 'deadline', 'ipk_min',
        'final_score', 'semantic_score', 'wsm_score', 'major_score'
    ]]
    print('\n=== TABEL HASIL REKOMENDASI ===')
    display(df_hasil)

    # ── 4. Export CSV lengkap ─────────────────────────────────
    df_export = pd.DataFrame(hasil)[[
        'nama_beasiswa', 'penyelenggara', 'deadline', 'ipk_min', 'bidang',
        'final_score', 'semantic_score', 'wsm_score', 'major_score', 'url'
    ]]
    df_export.to_csv('hasil_rekomendasi.csv', index=False, encoding='utf-8-sig')

    try:
        from google.colab import files
        files.download('hasil_rekomendasi.csv')
        print('\n📥 Hasil didownload sebagai hasil_rekomendasi.csv')
    except ImportError:
        print('\n✅ Hasil disimpan sebagai hasil_rekomendasi.csv')


---
## 📁 Referensi — Kode Asli (tidak perlu dijalankan)
> Cell-cell di bawah ini adalah kode pipeline lama (sebelum ditambah WSM).
> Dipertahankan sebagai referensi. **Gunakan eksekusi utama di atas.**

In [ ]:
import pandas as pd

df = pd.read_csv(
    "dataset_beasiswa.csv",
    sep=";"
)

print(df.columns.tolist())
display(df.head())

['id', 'nama_beasiswa', 'penyelenggara', 'jenjang', 'ipk_min', 'bidang', 'syarat', 'benefit', 'deadline', 'url']


,id,nama_beasiswa,penyelenggara,jenjang,ipk_min,bidang,syarat,benefit,deadline,url
0,B001,Djarum Beasiswa Plus 2026,Djarum Foundation,"D4,S1",3.0,Semua Jurusan,"Semester 4, IPK min 3.00 di sem 3, aktif organ...",Rp1.000.000/bulan selama 1 tahun + pelatihan s...,2026-06-10,https://djarumbeasiswaplus.org
1,B002,Beasiswa BCA Finance Peduli 2026,PT BCA Finance,"D4,S1",3.3,Semua Jurusan,"Semester 4-6, IPK min 3.30 (PTN) / 3.50 (PTS),...",Rp3.500.000/semester + pelatihan online persia...,2026-06-21,https://bcafinance.co.id
2,B003,Beasiswa Etos ID 2026,Dompet Dhuafa & GREAT Edunesia,S1,0.0,Semua Jurusan,"Mahasiswa baru PTN 2026 (jalur SNBP/SNBT), ber...",Bantuan UKT hingga 8 semester + uang saku Rp60...,2026-06-10,https://etosid.org
3,B004,Beasiswa DataPrint 2026 Periode 2,DataPrint Indonesia,"D3,D4,S1",0.0,Semua Jurusan,"Mahasiswa aktif D3/D4/S1, aktif organisasi, ti...",Rp500.000 (satu kali),2026-06-26,https://beasiswadataprint.com
4,B005,Beasiswa Glow and Lovely 2026,Glow and Lovely (Unilever),S1,0.0,Semua Jurusan,"Khusus perempuan, mahasiswa S1 aktif, WNI, pun...",Bantuan biaya kuliah + program mentoring dan p...,2026-06-30,https://kelasberbagicerah.com


In [ ]:
# ==========================================
# 2. DATA PROGRAM BEASISWA
# ==========================================
def load_scholarships_from_csv(csv_path="dataset_beasiswa.csv"):

    df = pd.read_csv(
        csv_path,
        sep=";"
    )

    df["matching_text"] = (
        df["nama_beasiswa"].fillna("") + " " +
        df["bidang"].fillna("") + " " +
        df["syarat"].fillna("") + " " +
        df["benefit"].fillna("")
    )

    scholarships = []

    for _, row in df.iterrows():

        scholarships.append({
            "id": row["id"],
            "nama": row["nama_beasiswa"],
            "syarat": row["matching_text"]
        })

    documents = [x["syarat"] for x in scholarships]

    print(f" Loaded {len(scholarships)} beasiswa")

    return scholarships, documents

In [ ]:
scholarships, documents = load_scholarships_from_csv(
    "dataset_beasiswa.csv"
)

print(len(scholarships))
print(len(documents))

 Loaded 6 beasiswa
6
6


In [ ]:
# ==========================================
# 3. OPSI 1: CROSS-ENCODER (IDEALIST)
# ==========================================
def run_cross_encoder_idealist(cv_text, scholarship_docs, scholarship_data):
    print("=== MENGJALANKAN OPSI 1: CROSS-ENCODER (IDEALIST) ===")
    model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

    pairs = [[cv_text, doc] for doc in scholarship_docs]
    scores = model.predict(pairs)

    results = []
    for i, score in enumerate(scores):
        results.append((scholarship_data[i]["nama"], score))

    results = sorted(results, key=lambda x: x[1], reverse=True)

    print("\n[Hasil Ranking Opsi 1]")
    for rank, (nama, score) in enumerate(results, 1):
        print(f"{rank}. {nama} (Skor Relevansi: {score:.4f})")
    print("-" * 50 + "\n")

In [ ]:
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity

def run_hybrid_realistic(cv_text, scholarship_docs, scholarship_data):
    print("=== MENJALANKAN OPSI 2: HYBRID BM25 + BI-ENCODER (REALISTIC) ===")

    # --- TAHAP 1: Lexical Search (BM25) ---
    # Tokenisasi sederhana (memecah kalimat menjadi kata)
    tokenized_docs = [doc.lower().split() for doc in scholarship_docs]
    bm25 = BM25Okapi(tokenized_docs)

    tokenized_cv = cv_text.lower().split()
    bm25_scores = bm25.get_scores(tokenized_cv)

    # Normalisasi skor BM25 ke rentang 0-1 (Min-Max Scaling) agar bisa digabungkan
    bm25_min, bm25_max = min(bm25_scores), max(bm25_scores)
    if bm25_max > bm25_min:
        bm25_norm = [(s - bm25_min) / (bm25_max - bm25_min) for s in bm25_scores]
    else:
        bm25_norm = [0] * len(bm25_scores)

    # --- TAHAP 2: Semantic Search (Bi-Encoder) ---
    # Model Bi-Encoder ini sangat cepat karena bisa melakukan pre-computation
    model = SentenceTransformer('all-MiniLM-L6-v2')

    # Dalam skenario production nyata, doc_embeddings ini sudah disimpan di Vector DB (FAISS/Milvus)
    doc_embeddings = model.encode(scholarship_docs)
    cv_embedding = model.encode([cv_text])

    # Hitung Cosine Similarity
    semantic_scores = cosine_similarity(cv_embedding, doc_embeddings)[0]

    # Normalisasi skor Semantik
    sem_min, sem_max = min(semantic_scores), max(semantic_scores)
    if sem_max > sem_min:
        semantic_norm = [(s - sem_min) / (sem_max - sem_min) for s in semantic_scores]
    else:
        semantic_norm = [0] * len(semantic_scores)

    # --- TAHAP 3: Score Fusion ---
    # Menggabungkan skor leksikal dan semantik (bisa diatur bobotnya, misal 30% kata kunci, 70% makna)
    alpha = 0.3 # Bobot Lexical
    beta = 0.7  # Bobot Semantic

    final_results = []
    for i in range(len(scholarship_data)):
        final_score = (alpha * bm25_norm[i]) + (beta * semantic_norm[i])
        final_results.append({
            "nama": scholarship_data[i]["nama"],
            "bm25_score": bm25_norm[i],
            "semantic_score": semantic_norm[i],
            "final_score": final_score
        })

    # Urutkan berdasarkan final_score
    final_results = sorted(final_results, key=lambda x: x["final_score"], reverse=True)

    print("\n[Hasil Ranking Opsi 2 (Hybrid)]")
    for rank, res in enumerate(final_results, 1):
        print(f"{rank}. {res['nama']}")
        print(f"   ↳ Skor Akhir: {res['final_score']:.4f} (BM25: {res['bm25_score']:.2f} | Semantik: {res['semantic_score']:.2f})")

# Cara memanggilnya nanti:
# run_hybrid_realistic(cv_user, documents, scholarships)